# EduAnalytics — Fase 3: Preparación de los datos
## Entradas reproducibles para clasificación multiclase · CRISP-DM

**Universidad:** Universidad Privada Franz Tamayo (UNIFRANZ). **Materia:** Minería de Datos.

**Integrantes**, según el README local:
- Milenka Melody Chuquimia Osco
- Edson Teddy Tito Chuquimia
- Carlos Daniel Valverde Mendoza
- Alvaro Luis Carlos del Carpio Blanco
- Marcelo Josue Escobar Chipana

**Propósito:** conservar los registros elegibles, definir predictores y objetivo, generar
particiones reproducibles y proporcionar preprocesamiento reutilizable para regresión logística
y árbol de decisión de la Fase 4. No se entrenan clasificadores ni se calculan métricas predictivas.

**Población y unidad:** cada fila representa un estudiante universitario según la documentación;
se estudian exclusivamente los estudiantes representados en el CSV. UNIFRANZ es la institución
del equipo, no una procedencia demostrada del dataset.

**Alcance:** retrospectivo sobre registros completos. Incluir GPA final y retención no acredita
disponibilidad anticipada. `Burnout_Risk_Level` no es un diagnóstico clínico. La procedencia
sintética no está confirmada y la fórmula de la etiqueta es desconocida: las asociaciones y
futuros resultados podrían reflejar reglas de generación, sin causalidad demostrada.

**Fuente:** `dataset/ai_student_impact_dataset (1).csv`, relativa a la raíz del proyecto.
**Salidas:** `wrangler/03_PREPARACION_DATOS/artefactos/`.

Ejecutar todas las celdas en orden desde un kernel limpio, desde la raíz o esta carpeta.
Las dependencias efectivas se documentan al exportar. Las rutas se derivan de la raíz encontrada,
sin rutas absolutas de un integrante.


## 1. Trazabilidad y continuidad con las Fases 1 y 2

Referencias: [README](../../README.md), [información general](../../docs/INFORMACION_GENERAL_PROYECTO_EDUANALYTICS.md),
[Fase 1](../01_ENTENDIMIENTO_NEGOCIO/01_entendimiento_negocio.ipynb) y
[Fase 2](../02_COMPRENSION_DATOS/02_comprension_datos_eda.ipynb).
Se verifica tanto la ruta declarada en su código como la huella guardada en sus salidas.
Una discrepancia detiene la exportación y deja explícito qué debe investigarse.


In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import ast
import csv
import hashlib
import importlib.metadata as metadata
import importlib.util
import io
import json
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted

assert sys.version_info[:3] == (3, 10, 6), "Usar Python 3.10.6."
RUTA_CSV = Path('dataset/ai_student_impact_dataset (1).csv')
RUTA_NOTEBOOK = Path('wrangler/03_PREPARACION_DATOS/03_preparacion_datos.ipynb')
RUTA_ARTEFACTOS = RUTA_NOTEBOOK.parent / 'artefactos'
RUTA_DOC = Path('docs/INFORMACION_GENERAL_PROYECTO_EDUANALYTICS.md')
PREVIOS = [Path('wrangler/01_ENTENDIMIENTO_NEGOCIO/01_entendimiento_negocio.ipynb'),
           Path('wrangler/02_COMPRENSION_DATOS/02_comprension_datos_eda.ipynb')]
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'README.md').is_file() and (p / 'docs').is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError('No se encontró la raíz. Ejecutar desde el repositorio o la carpeta del notebook.')
requeridos = [Path('README.md'), RUTA_DOC, *PREVIOS, RUTA_CSV, RUTA_NOTEBOOK]
faltantes = [p.as_posix() for p in requeridos if not (RAIZ / p).is_file()]
if faltantes:
    raise FileNotFoundError('Preparación pendiente: faltan ' + ', '.join(faltantes))

def sha256(contenido):
    return hashlib.sha256(contenido).hexdigest()

_numero_tabla = 0

def mostrar(datos, precision=3):
    global _numero_tabla
    _numero_tabla += 1
    display(datos.style.set_uuid(f'prep_{_numero_tabla}').format(precision=precision, decimal=',', thousands='.', na_rep='Pendiente')
            .set_properties(**{'text-align': 'left'})
            .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left'), ('vertical-align', 'top')]}]), exclude=['text/plain'])

def texto(mensaje):
    display(Markdown(mensaje))

def es(numero, decimales=3):
    return f'{numero:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')

documento = (RAIZ / RUTA_DOC).read_text(encoding='utf-8-sig')
readme = (RAIZ / 'README.md').read_text(encoding='utf-8-sig')
contenido_csv = (RAIZ / RUTA_CSV).read_bytes()
hash_entrada = sha256(contenido_csv)
referencias = []
for ruta in PREVIOS:
    previo = json.loads((RAIZ / ruta).read_text(encoding='utf-8-sig'))
    rutas = []
    for celda in previo['cells']:
        if celda['cell_type'] == 'code':
            fuente = ''.join(celda['source'])
            fuente_python = '\n'.join(linea for linea in fuente.splitlines() if not linea.lstrip().startswith('%'))
            for nodo in ast.walk(ast.parse(fuente_python)):
                if isinstance(nodo, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'RUTA_CSV' for t in nodo.targets):
                    rutas.append(Path(ast.literal_eval(nodo.value.args[0])).as_posix())
    salida = '\n'.join(''.join(o.get('text', [])) for c in previo['cells'] for o in c.get('outputs', []))
    huellas = sorted(set(re.findall(r'SHA-256(?: actual)?:\s*([a-f0-9]{64})', salida)))
    referencias.append({'Notebook': ruta.as_posix(), 'Ruta CSV declarada': '; '.join(rutas),
                        'SHA-256 guardado': '; '.join(huellas),
                        'Coincide': rutas == [RUTA_CSV.as_posix()] and huellas == [hash_entrada]})
mostrar(pd.DataFrame(referencias))
if not all(r['Coincide'] for r in referencias):
    raise ValueError('Preparación pendiente: ruta o huella no coincide con ambas fases. Investigar versiones antes de exportar.')

df_original = pd.read_csv(io.BytesIO(contenido_csv), sep=';', encoding='utf-8-sig', float_precision='round_trip')
lector = csv.DictReader(io.StringIO(contenido_csv.decode('utf-8-sig')), delimiter=';')
filas_crudas = list(lector)
if any(None in fila or any(v is None for v in fila.values()) for fila in filas_crudas):
    raise ValueError('Preparación pendiente: hay filas con cantidades de campos inconsistentes.')
crudo = pd.DataFrame(filas_crudas, columns=lector.fieldnames)
firma_original = pd.util.hash_pandas_object(df_original, index=True).values.copy()
columnas_originales = list(df_original.columns)
print('CSV:', RUTA_CSV.as_posix())
print('Separador: ; | Codificación: UTF-8 compatible con BOM')
print('Dimensiones:', df_original.shape)
print('SHA-256:', hash_entrada)
print('Referencia de KPI: porcentaje de registros High; no métrica predictiva.')

# Huella de contenido: complemento del SHA-256 estricto de bytes.
columnas_numericas_csv = set(df_original.select_dtypes(include='number').columns)
contenido_canonico = [list(lector.fieldnames), *[
    [format(Decimal(fila[c]).normalize(), 'f') if c in columnas_numericas_csv else fila[c]
     for c in lector.fieldnames] for fila in filas_crudas]]
hash_semantico = sha256(json.dumps(contenido_canonico, ensure_ascii=False, separators=(',', ':')).encode('utf-8'))
assert hash_semantico == '21b6b0658b1c909045bed401a821f417f82bb9f86795ff9f346473339379d23d', 'El contenido no equivale al original verificado.'


,Notebook,Ruta CSV declarada,SHA-256 guardado,Coincide
0,wrangler/01_ENTENDIMIENTO_NEGOCIO/01_entendimiento_negocio.ipynb,dataset/ai_student_impact_dataset (1).csv,585c433cf9b447df6812c76dbb6732d2bb663689a7b595a9a4b5d777fea991d5,True
1,wrangler/02_COMPRENSION_DATOS/02_comprension_datos_eda.ipynb,dataset/ai_student_impact_dataset (1).csv,585c433cf9b447df6812c76dbb6732d2bb663689a7b595a9a4b5d777fea991d5,True


CSV: dataset/ai_student_impact_dataset (1).csv
Separador: ; | Codificación: UTF-8 compatible con BOM
Dimensiones: (50000, 16)
SHA-256: 585c433cf9b447df6812c76dbb6732d2bb663689a7b595a9a4b5d777fea991d5
Referencia de KPI: porcentaje de registros High; no métrica predictiva.


In [2]:
CLASES = ['Low', 'Medium', 'High']
OBJETIVO = 'Burnout_Risk_Level'
conteos_iniciales = df_original[OBJETIVO].value_counts().reindex(CLASES, fill_value=0)
kpi_inicial = Decimal(int(conteos_iniciales['High'])) * Decimal(100) / Decimal(len(df_original))
# Las cantidades documentadas se leen del documento local, sin modificarlo.
bloque_clases = re.search(r'^## 16\..*?(?=^## \d+\.|\Z)', documento, flags=re.M | re.S).group(0)
clases_documentadas = {}
for linea in bloque_clases.splitlines():
    if linea.strip().startswith('|'):
        valores = [v.strip() for v in linea.strip().strip('|').split('|')]
        if valores[0] in CLASES:
            clases_documentadas[valores[0]] = int(valores[1].replace('.', ''))
continuidad = pd.DataFrame({'Cantidad documentada': pd.Series(clases_documentadas),
                           'Cantidad CSV': conteos_iniciales,
                           'Porcentaje CSV': 100 * conteos_iniciales / len(df_original)}).reindex(CLASES)
continuidad['Diferencia de cantidad'] = continuidad['Cantidad CSV'] - continuidad['Cantidad documentada']
mostrar(continuidad)
if set(clases_documentadas) != set(CLASES) or continuidad['Diferencia de cantidad'].ne(0).any():
    raise ValueError('Hay discrepancias en las clases documentadas; investigar antes de exportar, sin ajustar los datos.')
if df_original.shape != (50_000, 16) or kpi_inicial != Decimal('24.974'):
    raise ValueError('Dimensiones o KPI no reproducen las fases anteriores. Investigar antes de preparar.')
texto(f'**Continuidad comprobada:** {es(len(df_original), 0)} registros y {df_original.shape[1]} variables; '
      f'{es(conteos_iniciales.High, 0)} etiquetas High / {es(len(df_original), 0)} = '
      f'**{es(kpi_inicial)} %**, aproximadamente **{es(kpi_inicial, 2)} %**. '
      'Coinciden la huella, las clases y la línea base; el porcentaje es descriptivo del archivo.')


,Cantidad documentada,Cantidad CSV,Porcentaje CSV,Diferencia de cantidad
Low,16.369,16.369,"32,738",0
Medium,21.144,21.144,"42,288",0
High,12.487,12.487,"24,974",0


**Continuidad comprobada:** 50.000 registros y 16 variables; 12.487 etiquetas High / 50.000 = **24,974 %**, aproximadamente **24,97 %**. Coinciden la huella, las clases y la línea base; el porcentaje es descriptivo del archivo.

## 2. Calidad y decisiones de limpieza

Se vuelven a verificar los controles de Fase 2. Cumplir un rango plausible no demuestra que
la medición sea verdadera o clínicamente validada. Las normalizaciones de texto solo detectan
variantes, sin modificar valores. Si aparece una incidencia nueva, se detiene la exportación
para revisarla; no se inventa una regla de limpieza.


In [3]:
rangos = {
    'Pre_Semester_GPA': (0, 4, False, 'Escala GPA provisional'),
    'Post_Semester_GPA': (0, 4, False, 'Escala GPA provisional'),
    'Weekly_GenAI_Hours': (0, 168, False, 'Límite temporal h/semana'),
    'Traditional_Study_Hours': (0, 168, False, 'Límite temporal h/semana'),
    'Perceived_AI_Dependency': (1, 10, True, 'Escala citada en docs'),
    'Anxiety_Level_During_Exams': (1, 10, True, 'Escala citada en docs'),
    'Tool_Diversity': (1, 5, True, 'Conteo/rango citado en docs'),
    'Skill_Retention_Score': (0, 100, False, 'Puntos sobre 100 según docs'),
}
nulos = pd.DataFrame({'Nulos': df_original.isna().sum(),
                      '% nulos': 100 * df_original.isna().sum() / len(df_original),
                      'Campos vacíos/espacios': crudo.apply(lambda s: s.str.strip().eq('').sum())})
mostrar(nulos)
duplicados_completos = int(df_original.duplicated().sum())
duplicados_id = int(df_original.Student_ID.duplicated().sum())
objetivos_invalidos = ~df_original[OBJETIVO].isin(CLASES)
mascaras_rango = {}
resumen_rangos = []
for c, (lo, hi, entero, razon) in rangos.items():
    s = df_original[c]
    fuera = s.notna() & (~s.between(lo, hi) | ~np.isfinite(s))
    no_entero = s.notna() & s.mod(1).ne(0) if entero else pd.Series(False, index=s.index)
    mascaras_rango[c] = fuera | no_entero
    resumen_rangos.append({'Variable': c, 'Límite inferior': lo, 'Límite superior': hi,
                           'Base del criterio': razon, 'Fuera de rango/no finitos': int(fuera.sum()),
                           'No enteros cuando corresponde': int(no_entero.sum())})
mostrar(pd.DataFrame(resumen_rangos))

categoricas_crudas = [c for c in columnas_originales if c not in rangos and c != 'Student_ID']
auditoria_categorias = []
variantes = []
def canonica(valor):
    return re.sub(r'[\s_/]+', ' ', unicodedata.normalize('NFKC', valor).strip().casefold())
for c in categoricas_crudas:
    s = crudo[c]
    grupos_variantes = {}
    for valor in s.drop_duplicates():
        grupos_variantes.setdefault(canonica(valor), []).append(valor)
    colisiones = [v for v in grupos_variantes.values() if len(v) > 1]
    auditoria_categorias.append({'Variable': c, 'Categorías originales': '; '.join(sorted(s.unique())),
                                'Filas con espacios extremos': int(s.ne(s.str.strip()).sum()),
                                'Grupos de variantes internas': len(colisiones)})
    for valores in colisiones:
        variantes.append({'Variable': c, 'Variantes': repr(valores), 'Filas': int(s.isin(valores).sum())})
mostrar(pd.DataFrame(auditoria_categorias))
if variantes:
    mostrar(pd.DataFrame(variantes))

calidad = {
    'Celdas nulas': int(nulos.Nulos.sum()), 'Campos vacíos/espacios': int(nulos['Campos vacíos/espacios'].sum()),
    'Duplicados completos excedentes': duplicados_completos, 'IDs repetidos excedentes': duplicados_id,
    'Objetivos inválidos': int(objetivos_invalidos.sum()),
    'Incidencias de rango': sum(int(m.sum()) for m in mascaras_rango.values()),
    'Filas con espacios extremos': sum(v['Filas con espacios extremos'] for v in auditoria_categorias),
    'Grupos de variantes internas': sum(v['Grupos de variantes internas'] for v in auditoria_categorias),
}
mostrar(pd.DataFrame.from_dict(calidad, orient='index', columns=['Cantidad']), 0)
if any(calidad.values()):
    if duplicados_id:
        display(df_original.loc[df_original.Student_ID.duplicated(keep=False), ['Student_ID', OBJETIVO]].head(20))
    raise ValueError('Incidencias nuevas de calidad: exportación detenida. Si hay varios registros por estudiante, '
                     'es necesaria una división por grupos; no se dividirán filas del mismo estudiante entre conjuntos.')
texto('**Resultado:** no se requieren correcciones de registros. Se mantienen valores y categorías; '
      'no se imputan, deduplican, eliminan ni recortan observaciones. Las diferencias entre categorías de docs '
      'y el CSV corresponden al diccionario documental, no a variantes coexistentes detectadas en el archivo.')


,Nulos,% nulos,Campos vacíos/espacios
Student_ID,0,"0,000",0
Major_Category,0,"0,000",0
Year_of_Study,0,"0,000",0
Pre_Semester_GPA,0,"0,000",0
Weekly_GenAI_Hours,0,"0,000",0
Primary_Use_Case,0,"0,000",0
Prompt_Engineering_Skill,0,"0,000",0
Tool_Diversity,0,"0,000",0
Paid_Subscription,0,"0,000",0
Traditional_Study_Hours,0,"0,000",0


,Variable,Límite inferior,Límite superior,Base del criterio,Fuera de rango/no finitos,No enteros cuando corresponde
0,Pre_Semester_GPA,0,4,Escala GPA provisional,0,0
1,Post_Semester_GPA,0,4,Escala GPA provisional,0,0
2,Weekly_GenAI_Hours,0,168,Límite temporal h/semana,0,0
3,Traditional_Study_Hours,0,168,Límite temporal h/semana,0,0
4,Perceived_AI_Dependency,1,10,Escala citada en docs,0,0
5,Anxiety_Level_During_Exams,1,10,Escala citada en docs,0,0
6,Tool_Diversity,1,5,Conteo/rango citado en docs,0,0
7,Skill_Retention_Score,0,100,Puntos sobre 100 según docs,0,0


,Variable,Categorías originales,Filas con espacios extremos,Grupos de variantes internas
0,Major_Category,Arts; Business; Humanities; Medical; STEM,0,0
1,Year_of_Study,Freshman; Graduate; Junior; Senior; Sophomore,0,0
2,Primary_Use_Case,Copywriting/Drafting; Debugging/Troubleshooting; Direct_Answer_Generation; Ideation; Summarizing_Reading,0,0
3,Prompt_Engineering_Skill,Advanced; Beginner; Intermediate,0,0
4,Paid_Subscription,False; True,0,0
5,Institutional_Policy,Actively_Encouraged; Allowed_With_Citation; Strict_Ban,0,0
6,Burnout_Risk_Level,High; Low; Medium,0,0


,Cantidad
Celdas nulas,0
Campos vacíos/espacios,0
Duplicados completos excedentes,0
IDs repetidos excedentes,0
Objetivos inválidos,0
Incidencias de rango,0
Filas con espacios extremos,0
Grupos de variantes internas,0


**Resultado:** no se requieren correcciones de registros. Se mantienen valores y categorías; no se imputan, deduplican, eliminan ni recortan observaciones. Las diferencias entre categorías de docs y el CSV corresponden al diccionario documental, no a variantes coexistentes detectadas en el archivo.

In [4]:
decisiones = pd.DataFrame([
    ('No se observan nulos', 'Todas las columnas', 'Conservar; no imputar', 'No existe ausencia que corregir', 0, 0),
    ('No se observan duplicados completos', 'Todas las filas', 'Conservar; no deduplicar', 'Cada registro es distinto', 0, 0),
    ('No se repite Student_ID', 'Student_ID', 'División estratificada por filas', 'Un registro por identificador en el CSV actual', 0, 0),
    ('Sin variantes internas ni espacios extremos', 'Variables categóricas', 'Mantener etiquetas originales', 'No se requiere normalización de registros', 0, 0),
    ('Categorías alineadas con docs', 'Primary_Use_Case', 'Conservar nombres exactos', 'Summarizing_Reading y Direct_Answer_Generation son los nombres del CSV', 0, 0),
    ('Sin incumplimientos de rangos declarados', 'Ocho variables numéricas', 'Conservar', 'No hay evidencia de valores fuera de los criterios provisionales', 0, 0),
    ('Extremos IQR plausibles señalados en Fase 2', 'Horas de IA/estudio, dependencia, GPA y retención', 'Conservar; sin winsorización ni eliminación', 'Un extremo no es automáticamente un error; posible interés para High', 0, 0),
    ('Etiqueta válida', 'Burnout_Risk_Level', 'Conservar Low, Medium y High como texto', 'No requiere codificación numérica para los modelos previstos', 0, 0),
], columns=['Problema o resultado del control', 'Variables/registros afectados', 'Regla aplicada',
            'Justificación', 'Registros modificados', 'Registros excluidos'])
mostrar(decisiones)
df_elegible = df_original.copy(deep=True)
assert df_elegible.equals(df_original)
texto('**Decisión de limpieza:** cero registros modificados y cero excluidos. '
      'Los extremos IQR de la Fase 2 permanecen en el conjunto elegible; no se aplica un límite aprendido del dataset completo. '
      'Si la Fase 4 evaluara recortes u otros parámetros aprendidos, deberán ajustarse dentro del entrenamiento de cada pliegue.')


,Problema o resultado del control,Variables/registros afectados,Regla aplicada,Justificación,Registros modificados,Registros excluidos
0,No se observan nulos,Todas las columnas,Conservar; no imputar,No existe ausencia que corregir,0,0
1,No se observan duplicados completos,Todas las filas,Conservar; no deduplicar,Cada registro es distinto,0,0
2,No se repite Student_ID,Student_ID,División estratificada por filas,Un registro por identificador en el CSV actual,0,0
3,Sin variantes internas ni espacios extremos,Variables categóricas,Mantener etiquetas originales,No se requiere normalización de registros,0,0
4,Categorías alineadas con docs,Primary_Use_Case,Conservar nombres exactos,Summarizing_Reading y Direct_Answer_Generation son los nombres del CSV,0,0
5,Sin incumplimientos de rangos declarados,Ocho variables numéricas,Conservar,No hay evidencia de valores fuera de los criterios provisionales,0,0
6,Extremos IQR plausibles señalados en Fase 2,"Horas de IA/estudio, dependencia, GPA y retención",Conservar; sin winsorización ni eliminación,Un extremo no es automáticamente un error; posible interés para High,0,0
7,Etiqueta válida,Burnout_Risk_Level,"Conservar Low, Medium y High como texto",No requiere codificación numérica para los modelos previstos,0,0


**Decisión de limpieza:** cero registros modificados y cero excluidos. Los extremos IQR de la Fase 2 permanecen en el conjunto elegible; no se aplica un límite aprendido del dataset completo. Si la Fase 4 evaluara recortes u otros parámetros aprendidos, deberán ajustarse dentro del entrenamiento de cada pliegue.

## 3. Predictores, objetivo y funciones reutilizables

Se incluyen **14 predictores** del registro completo, incluidos GPA final y retención por el
alcance retrospectivo acordado. Se excluyen ID y etiqueta. La función `crear_preprocesador`
siempre devuelve un pipeline nuevo, sin parámetros aprendidos.

La siguiente celda contiene el código completo que se exportará como `artefactos/preprocesamiento.py`.
La carga devuelve `(X, y, trazabilidad)` con índice `_row_id` compartido. La columna booleana
se valida mediante las cadenas exactas `True` y `False`; no se usa la verdad de una cadena no vacía.
Los CSV se leen con `float_precision='round_trip'` para conservar los valores exportados.


In [5]:
from pathlib import Path
import hashlib
import io
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

CLASES = ['Low', 'Medium', 'High']
OBJETIVO = 'Burnout_Risk_Level'
NUMERICAS = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity', 'Traditional_Study_Hours',
            'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']
CATEGORICAS = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Institutional_Policy']
BOOLEANAS = ['Paid_Subscription']
PREDICTORES = ['Major_Category', 'Year_of_Study', 'Pre_Semester_GPA', 'Weekly_GenAI_Hours',
              'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Tool_Diversity', 'Paid_Subscription',
              'Traditional_Study_Hours', 'Perceived_AI_Dependency', 'Institutional_Policy',
              'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']
TRAZABILIDAD = ['_row_id', 'Student_ID']

# Referencia congelada antes de corregir el proyecto; incluye orden de cada muestra.
HUELLAS_PARTICIONES = {'train': 'b8ceb4a646692d3ed92917fd753b5cc155a364eac5db5a69a83f6478f80660da', 'validation': 'd0739661c229bead590deabdf7a4c6da492001c825fc09f5c186b6c21d13e9ac', 'test': 'ee8776d4e6c10987aed087aecf017a6d8298978b17120ce24716bd81b1a2818e'}

def huella_pertenencia(datos):
    secuencia = datos[TRAZABILIDAD].to_numpy(dtype=np.int64).tolist()
    return hashlib.sha256(json.dumps(secuencia, separators=(',', ':')).encode('utf-8')).hexdigest()

def crear_preprocesador(modelo):
    """Pipeline nuevo sin ajustar; no contiene clasificador ni usa etiquetas."""
    if modelo not in {'logistica', 'arbol'}:
        raise ValueError("Modelo debe ser 'logistica' o 'arbol'.")
    numerico = Pipeline([('escalado', StandardScaler())]) if modelo == 'logistica' else 'passthrough'
    columnas = ColumnTransformer([
        ('numericas', numerico, NUMERICAS),
        ('categoricas', OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), CATEGORICAS),
        ('booleanas', FunctionTransformer(np.asarray, kw_args={'dtype': np.float64},
                                         feature_names_out='one-to-one'), BOOLEANAS),
    ], remainder='drop', sparse_threshold=0)
    return Pipeline([('columnas', columnas)])

def cargar_particion(nombre, carpeta):
    """Carga una partición original y verifica hash, esquema, tipos, tamaño y clases."""
    if nombre not in {'train', 'validation', 'test'}:
        raise ValueError("Partición debe ser 'train', 'validation' o 'test'.")
    carpeta = Path(carpeta)
    manifiesto = json.loads((carpeta / 'manifest.json').read_text(encoding='utf-8'))
    if manifiesto['schema_version'] != 1 or manifiesto['schema']['predictors'] != PREDICTORES:
        raise ValueError('Manifiesto incompatible con las funciones de preparación.')
    informacion = manifiesto['partitions'][nombre]
    if informacion['file'] != f'{nombre}.csv':
        raise ValueError('Nombre de archivo de partición inesperado.')
    contenido = (carpeta / informacion['file']).read_bytes()
    if hashlib.sha256(contenido).hexdigest() != manifiesto['artifacts'][informacion['file']]['sha256']:
        raise ValueError(f'Hash incorrecto en {informacion["file"]}; no se utiliza una partición modificada.')
    tipos = dict(manifiesto['schema']['original_dtypes'])
    tipos.update({'_row_id': 'int64', 'Paid_Subscription': 'string'})
    datos = pd.read_csv(io.BytesIO(contenido), sep=';', encoding='utf-8', dtype=tipos,
                        float_precision='round_trip')
    if list(datos.columns) != manifiesto['schema']['partition_columns']:
        raise ValueError('Columnas o su orden no coinciden con el manifiesto.')
    if datos['Paid_Subscription'].isna().any() or not datos['Paid_Subscription'].isin(['True', 'False']).all():
        raise ValueError('Paid_Subscription contiene un valor distinto de True/False.')
    datos['Paid_Subscription'] = datos['Paid_Subscription'].map({'True': True, 'False': False}).astype(bool)
    if datos.isna().any().any() or not np.isfinite(datos[NUMERICAS].to_numpy()).all():
        raise ValueError('La partición no cumple la calidad sin nulos/no finitos registrada.')
    if len(datos) != informacion['rows'] or not datos['_row_id'].is_unique or not datos['Student_ID'].is_unique:
        raise ValueError('Tamaño o identificadores de la partición inconsistentes.')
    if not datos[OBJETIVO].isin(CLASES).all():
        raise ValueError('Objetivo con clases inesperadas.')
    conteos = {c: int(datos[OBJETIVO].eq(c).sum()) for c in CLASES}
    if conteos != informacion['class_counts'] or not all(conteos.values()):
        raise ValueError('Distribución o presencia de clases no coincide con el manifiesto.')
    if huella_pertenencia(datos) != HUELLAS_PARTICIONES[nombre]:
        raise ValueError('Pertenencia u orden distintos de la partición congelada.')
    if manifiesto['split']['membership_sha256'][nombre] != HUELLAS_PARTICIONES[nombre]:
        raise ValueError('Huella de pertenencia del manifiesto inconsistente.')
    datos.index = pd.Index(datos['_row_id'].to_numpy(), name='_row_id')
    X = datos[PREDICTORES].copy()
    y = datos[OBJETIVO].copy()
    trazabilidad = datos[TRAZABILIDAD].copy()
    assert X.index.equals(y.index) and X.index.equals(trazabilidad.index)
    return X, y, trazabilidad


In [6]:
assert set(df_elegible.columns) == set(PREDICTORES + ['Student_ID', OBJETIVO])
copias_directas = []
for c in PREDICTORES:
    copia_texto = df_elegible[c].astype(str).equals(df_elegible[OBJETIVO].astype(str))
    codigo_biyectivo = (df_elegible[c].nunique(dropna=False) == len(CLASES)
                        and df_elegible.groupby(OBJETIVO)[c].nunique(dropna=False).eq(1).all())
    if copia_texto or codigo_biyectivo:
        copias_directas.append(c)
print('Copias textuales o codificaciones biyectivas detectadas:', copias_directas)
if copias_directas:
    raise ValueError('Fuga directa: revisar y excluir las columnas detectadas antes de exportar.')

variables = []
for c in columnas_originales:
    if c in NUMERICAS:
        transformacion = 'StandardScaler en logística; valores originales en árbol'
        justificacion = ('Medición del registro completo; inclusión retrospectiva, revisar disponibilidad futura'
                        if c in ['Post_Semester_GPA', 'Skill_Retention_Score', 'Anxiety_Level_During_Exams']
                        else 'Característica académica o hábito pertinente al objetivo')
        papel = 'Predictor numérico' + (' (puntuación ordinal)' if c in ['Perceived_AI_Dependency', 'Anxiety_Level_During_Exams'] else '')
    elif c in CATEGORICAS:
        transformacion = 'One-hot: categorías aprendidas solo con entrenamiento'
        justificacion = ('Evitar distancias artificiales y ambigüedad de Graduate'
                        if c in ['Year_of_Study', 'Prompt_Engineering_Skill'] else 'Contexto o finalidad nominal sin orden numérico')
        papel = 'Predictor categórico'
    elif c in BOOLEANAS:
        transformacion, justificacion, papel = 'False→0; True→1', 'Indicador binario del registro', 'Predictor booleano'
    else:
        transformacion = 'Sin transformación predictora'
        justificacion = 'Identificador sin significado sustantivo' if c == 'Student_ID' else 'Etiqueta objetivo; fuga directa si se incluye en X'
        papel = 'Trazabilidad' if c == 'Student_ID' else 'Objetivo multiclase'
    variables.append({'Variable': c, 'Tipo observado': str(df_original[c].dtype), 'Incluida en X': c in PREDICTORES,
                      'Papel': papel, 'Transformación prevista': transformacion, 'Justificación': justificacion})
variables.extend([
    {'Variable': '_row_id', 'Tipo observado': 'int64 (auxiliar)', 'Incluida en X': False, 'Papel': 'Trazabilidad',
     'Transformación prevista': 'Posición de fila original desde cero', 'Justificación': 'Rastrear cobertura y pertenencia sin usarla como predictor'},
    {'Variable': 'GPA_Change', 'Tipo observado': 'No incorporada', 'Incluida en X': False, 'Papel': 'Propuesta académica secundaria',
     'Transformación prevista': 'Post_Semester_GPA − Pre_Semester_GPA; no aplicada al conjunto principal',
     'Justificación': 'Evitar redundancia con ambos GPA y ampliar el alcance a regresión o clustering'},
])
variables = pd.DataFrame(variables)
mostrar(variables)
df_elegible['_row_id'] = np.arange(len(df_elegible), dtype=np.int64)
X = df_elegible[PREDICTORES].copy()
y = df_elegible[OBJETIVO].copy()
assert X.index.equals(y.index)
assert 'Student_ID' not in X and OBJETIVO not in X and '_row_id' not in X and 'GPA_Change' not in X
print('X:', X.shape, '| y:', y.shape, '| clases:', CLASES)
texto('**Selección:** 14 predictores, objetivo textual y dos campos de trazabilidad fuera de X. '
      'No se detectan copias directas de la etiqueta en el esquema actual; esto no demuestra independencia respecto '
      'de una fórmula sintética desconocida. Las puntuaciones de dependencia y ansiedad conservan su escala numérica '
      'observada, cuya interpretación de intervalos permanece pendiente de validación.')


Copias textuales o codificaciones biyectivas detectadas: []


,Variable,Tipo observado,Incluida en X,Papel,Transformación prevista,Justificación
0,Student_ID,int64,False,Trazabilidad,Sin transformación predictora,Identificador sin significado sustantivo
1,Major_Category,object,True,Predictor categórico,One-hot: categorías aprendidas solo con entrenamiento,Contexto o finalidad nominal sin orden numérico
2,Year_of_Study,object,True,Predictor categórico,One-hot: categorías aprendidas solo con entrenamiento,Evitar distancias artificiales y ambigüedad de Graduate
3,Pre_Semester_GPA,float64,True,Predictor numérico,StandardScaler en logística; valores originales en árbol,Característica académica o hábito pertinente al objetivo
4,Weekly_GenAI_Hours,float64,True,Predictor numérico,StandardScaler en logística; valores originales en árbol,Característica académica o hábito pertinente al objetivo
5,Primary_Use_Case,object,True,Predictor categórico,One-hot: categorías aprendidas solo con entrenamiento,Contexto o finalidad nominal sin orden numérico
6,Prompt_Engineering_Skill,object,True,Predictor categórico,One-hot: categorías aprendidas solo con entrenamiento,Evitar distancias artificiales y ambigüedad de Graduate
7,Tool_Diversity,int64,True,Predictor numérico,StandardScaler en logística; valores originales en árbol,Característica académica o hábito pertinente al objetivo
8,Paid_Subscription,bool,True,Predictor booleano,False→0; True→1,Indicador binario del registro
9,Traditional_Study_Hours,float64,True,Predictor numérico,StandardScaler en logística; valores originales en árbol,Característica académica o hábito pertinente al objetivo


X: (50000, 14) | y: (50000,) | clases: ['Low', 'Medium', 'High']


**Selección:** 14 predictores, objetivo textual y dos campos de trazabilidad fuera de X. No se detectan copias directas de la etiqueta en el esquema actual; esto no demuestra independencia respecto de una fórmula sintética desconocida. Las puntuaciones de dependencia y ansiedad conservan su escala numérica observada, cuya interpretación de intervalos permanece pendiente de validación.

### 3.1 Tres riesgos diferentes y propuesta académica secundaria

| Situación | Decisión y límite |
|---|---|
| Fuga directa | Etiqueta, sus copias/códigos y campos de trazabilidad quedan fuera de X. El esquema permitido está explícito. |
| Disponibilidad temporal | GPA final y retención se incluyen porque están en el registro completo retrospectivo. No se presentan como disponibles antes del cierre del semestre. Ansiedad y hábitos también requieren documentar momentos de medición. |
| Posible circularidad | Se desconoce si la etiqueta fue generada a partir de horas, dependencia, ansiedad u otros campos. No se afirma una fórmula. Evaluar sensibilidad en Fase 4, incluyendo comparación sin GPA final y retención, conservando las mismas particiones. |

`GPA_Change = Post_Semester_GPA - Pre_Semester_GPA` expresa el cambio académico en puntos;
se conserva como propuesta secundaria. No se añade a X ni a los CSV, porque el conjunto principal
ya incluye ambos GPA y aquí no se prepara regresión ni clustering. No se incorporan otras características derivadas.


## 4. Particiones reproducibles antes de transformar

Se separan 70 % de entrenamiento y 30 % restante mediante `train_test_split`; el restante se
divide por mitades en validación y prueba. Ambas operaciones usan semilla **42** y estratificación
por la etiqueta. Se preserva el orden devuelto por las operaciones, sin volver a barajar al cargar.

La unicidad de Student_ID ya fue comprobada. Si se repitiera, la auditoría detendría la exportación:
dividir por filas podría colocar una misma persona en distintos conjuntos y requeriría una
estrategia por grupos. La estratificación describe etiquetas existentes y no entrena un modelo.


In [7]:
SEMILLA = 42
PROPORCIONES = {'train': 0.70, 'validation': 0.15, 'test': 0.15}
indices_train, indices_restantes = train_test_split(df_elegible.index.to_numpy(), test_size=.30,
                                                  random_state=SEMILLA, stratify=y)
indices_validation, indices_test = train_test_split(indices_restantes, test_size=.50,
                                                  random_state=SEMILLA, stratify=y.loc[indices_restantes])
indices = {'train': indices_train, 'validation': indices_validation, 'test': indices_test}
particiones = {nombre: df_elegible.loc[idx].copy() for nombre, idx in indices.items()}
resumen_particiones = []
for nombre, datos in particiones.items():
    fila = {'Partición': nombre, 'Registros': len(datos), '% total': 100 * len(datos)/len(df_elegible)}
    for clase in CLASES:
        n = int(datos[OBJETIVO].eq(clase).sum())
        fila[clase] = n
        fila[f'% {clase}'] = 100 * n / len(datos)
    resumen_particiones.append(fila)
resumen_particiones = pd.DataFrame(resumen_particiones).set_index('Partición')
mostrar(resumen_particiones)

comprobaciones_particiones = []
for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    interseccion_filas = set(particiones[a]._row_id) & set(particiones[b]._row_id)
    interseccion_estudiantes = set(particiones[a].Student_ID) & set(particiones[b].Student_ID)
    assert not interseccion_filas and not interseccion_estudiantes
    comprobaciones_particiones.append({'Control': f'Intersección {a} / {b}',
                                      'Filas compartidas': len(interseccion_filas),
                                      'Estudiantes compartidos': len(interseccion_estudiantes)})
cobertura = np.concatenate([datos._row_id.to_numpy() for datos in particiones.values()])
assert len(cobertura) == len(df_elegible) and len(set(cobertura)) == len(df_elegible)
assert set(cobertura) == set(df_elegible._row_id)
for nombre, datos in particiones.items():
    assert set(datos[OBJETIVO]) == set(CLASES)
    assert X.loc[datos.index].index.equals(y.loc[datos.index].index)
    assert X.loc[datos.index].equals(datos[PREDICTORES])
    assert y.loc[datos.index].equals(datos[OBJETIVO])
    assert len(datos) == int(PROPORCIONES[nombre] * len(df_elegible))
mostrar(pd.DataFrame(comprobaciones_particiones), 0)
print('Cobertura:', len(cobertura), '| Filas distintas:', len(set(cobertura)), '| Tres clases presentes en cada conjunto: True')
texto('**Resultado:** entrenamiento, validación y prueba son disjuntos por fila y estudiante y cubren todos los registros elegibles. '
      'Los porcentajes por clase son prácticamente los originales; las pequeñas diferencias responden al reparto de cantidades enteras. '
      'Prueba se reserva para evaluación final de la Fase 4: no se utiliza para elegir predictores, transformaciones o hiperparámetros.')

# Bloquear cambios de pertenencia u orden ANTES de cualquier exportación.
for nombre, datos in particiones.items():
    assert huella_pertenencia(datos) == HUELLAS_PARTICIONES[nombre], 'La división difiere de la referencia congelada.'
existentes = {nombre: RAIZ / RUTA_ARTEFACTOS / f'{nombre}.csv' for nombre in particiones}
if any(p.exists() for p in existentes.values()):
    if not all(p.is_file() for p in existentes.values()):
        raise ValueError('Particiones existentes incompletas: investigar antes de reemplazarlas.')
    for nombre, ruta in existentes.items():
        existente = pd.read_csv(ruta, sep=';', encoding='utf-8', float_precision='round_trip')
        assert huella_pertenencia(existente) == HUELLAS_PARTICIONES[nombre]
        pd.testing.assert_frame_equal(existente.reset_index(drop=True), particiones[nombre].reset_index(drop=True), check_exact=True)
print('Pertenencia, valores y orden originales comprobados antes de exportar.')


,Registros,% total,Low,% Low,Medium,% Medium,High,% High
Partición,,,,,,,,
train,35.000,"70,000",11.458,"32,737",14.801,"42,289",8.741,"24,974"
validation,7.500,"15,000",2.455,"32,733",3.172,"42,293",1.873,"24,973"
test,7.500,"15,000",2.456,"32,747",3.171,"42,280",1.873,"24,973"


,Control,Filas compartidas,Estudiantes compartidos
0,Intersección train / validation,0,0
1,Intersección train / test,0,0
2,Intersección validation / test,0,0


Cobertura: 50000 | Filas distintas: 50000 | Tres clases presentes en cada conjunto: True


**Resultado:** entrenamiento, validación y prueba son disjuntos por fila y estudiante y cubren todos los registros elegibles. Los porcentajes por clase son prácticamente los originales; las pequeñas diferencias responden al reparto de cantidades enteras. Prueba se reserva para evaluación final de la Fase 4: no se utiliza para elegir predictores, transformaciones o hiperparámetros.

Pertenencia, valores y orden originales comprobados antes de exportar.


## 5. Diseño del preprocesamiento para cada modelo

| Bloque | Regresión logística | Árbol de decisión |
|---|---|---|
| Ocho variables numéricas | `StandardScaler`: media y DE aprendidas solo con entrenamiento | Valores originales; el árbol no requiere uniformar escalas para sus cortes |
| Cinco categóricas | One-hot sin eliminar categorías; categorías aprendidas en entrenamiento | La misma codificación nominal |
| Suscripción pagada | False→0, True→1, conversión sin parámetros aprendidos | Igual |
| Desconocidas | `handle_unknown='ignore'`: bloque de ceros para esa variable; documentar frecuencia en Fase 4 | Igual |
| Nulos | No imputación: auditoría y carga comprueban ausencia | Igual |
| Balanceo | No SMOTE, sobremuestreo ni pesos automáticos | Igual |

One-hot también se usa para `Year_of_Study` y `Prompt_Engineering_Skill`, según la decisión
acordada: evita distancias artificiales y la ambigüedad de Graduate. Dependencia y ansiedad son
puntuaciones ordinales almacenadas numéricamente; usarlas en logística implica una relación
lineal en su puntuación que deberá revisarse en el modelado.

La salida densa es apropiada para las pocas categorías actuales. La cantidad exacta de columnas
se aprende de entrenamiento, no se fija utilizando categorías de validación o prueba.
Los pipelines exportados son fábricas de instancias sin ajustar; no se guardan matrices transformadas.


In [8]:
preprocesadores = {nombre: crear_preprocesador(nombre) for nombre in ['logistica', 'arbol']}
for nombre, pipeline in preprocesadores.items():
    display(pipeline)
    assert not hasattr(pipeline.named_steps['columnas'], 'transformers_')
print('Ambas instancias están sin ajustar. Todavía no se calculan medias ni categorías del entrenamiento.')


,steps,"[('columnas', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numericas', ...), ('categoricas', ...), ...]"
,remainder,'drop'
,sparse_threshold,0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


,steps,"[('columnas', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numericas', ...), ('categoricas', ...), ...]"
,remainder,'drop'
,sparse_threshold,0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Ambas instancias están sin ajustar. Todavía no se calculan medias ni categorías del entrenamiento.


## 6. Exportación de entradas originales y documentos de traspaso

Los CSV se guardan **antes** de ajustar las copias diagnósticas de preprocesamiento.
Cada partición contiene las 16 columnas originales y `_row_id`; la carga separa X, y y
trazabilidad. Se verifica el contenido después de volver a leerlo.

El manifiesto describe entrada, referencias, esquema, procedimiento, versiones y decisiones.
Los hashes permiten comprobar que los notebooks de Fase 4 utilizan los mismos archivos.


In [9]:
carpeta_artefactos = RAIZ / RUTA_ARTEFACTOS
carpeta_artefactos.mkdir(parents=True, exist_ok=True)
for nombre, datos in particiones.items():
    datos.to_csv(carpeta_artefactos / f'{nombre}.csv', sep=';', index=False, encoding='utf-8', lineterminator='\n')
variables.to_csv(carpeta_artefactos / 'variables.csv', sep=';', index=False, encoding='utf-8', lineterminator='\n')
decisiones.to_csv(carpeta_artefactos / 'decisiones_limpieza.csv', sep=';', index=False, encoding='utf-8', lineterminator='\n')

# Exportar literalmente la celda visible y autosuficiente de funciones; no hay lógica oculta.
notebook_actual = json.loads((RAIZ / RUTA_NOTEBOOK).read_text(encoding='utf-8'))
celdas_modulo = [c for c in notebook_actual['cells'] if 'modulo_reutilizable' in c.get('metadata', {}).get('tags', [])]
assert len(celdas_modulo) == 1
fuente_modulo = ''.join(celdas_modulo[0]['source']).strip() + '\n'
(carpeta_artefactos / 'preprocesamiento.py').write_text(fuente_modulo, encoding='utf-8', newline='\n')
paquetes = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'ipykernel', 'nbformat', 'nbclient', 'nbconvert', 'matplotlib', 'seaborn', 'ipython']
versiones = {p: metadata.version(p) for p in paquetes}
(carpeta_artefactos / 'requirements_preparacion.txt').write_text(
    '# Python 3.10.6; mismas versiones que el entorno principal.\n-r ../../../requirements.txt\n',
    encoding='utf-8', newline='\n')


89

In [10]:
instrucciones_fase4 = r"""# EduAnalytics: carga y modelado posterior a Fase 3

Estos archivos corresponden a clasificación retrospectiva multiclase de Burnout_Risk_Level.
Las etiquetas no son diagnósticos clínicos. Publicación Kaggle, versión 1 y licencia CC0 verificadas; fórmula, instrumentos y origen sintético pendientes. Véase docs/PROCEDENCIA_DATASET.md.

## Archivos

- train.csv, validation.csv y test.csv: registros originales separados por ;, UTF-8, con _row_id.
- variables.csv y decisiones_limpieza.csv: selección y reglas documentadas.
- manifest.json: huella de entrada, referencias, procedimiento, esquema, clases y hashes de salidas.
- preprocesamiento.py: crear_preprocesador(modelo) y cargar_particion(nombre, carpeta).
- requirements_preparacion.txt: versiones verificadas; Python 3.10.6 en esta ejecución.

## Carga desde un notebook de Fase 4

Ejecutar desde la raíz del proyecto o una carpeta dentro de él. Instalar las versiones indicadas
en un entorno virtual compatible cuando sea necesario: python -m pip install -r
wrangler/03_PREPARACION_DATOS/artefactos/requirements_preparacion.txt.

```python
from pathlib import Path
import sys
from sklearn.pipeline import Pipeline

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'README.md').is_file() and (p / 'wrangler').is_dir())
artefactos = raiz / 'wrangler/03_PREPARACION_DATOS/artefactos'
sys.path.insert(0, str(artefactos))
from preprocesamiento import cargar_particion, crear_preprocesador

X_train, y_train, traza_train = cargar_particion('train', artefactos)
X_val, y_val, traza_val = cargar_particion('validation', artefactos)
preprocesador_logistica = crear_preprocesador('logistica')
preprocesador_arbol = crear_preprocesador('arbol')
```

X contiene los mismos 14 predictores en el mismo orden. y conserva Low, Medium y High como texto.
La trazabilidad contiene _row_id y Student_ID. No añadir esas columnas ni el objetivo a X.
El lector verifica el hash, esquema, tamaño y clases de cada partición antes de entregarla.

## Ajuste correcto dentro de Fase 4

Crear el clasificador en la Fase 4 e integrarlo en un pipeline completo:

```python
# modelo_fase4 será una regresión logística o un árbol definidos en la Fase 4.
pipeline_completo = Pipeline([
    ('preprocesamiento', crear_preprocesador('logistica')),  # 'arbol' para el árbol
    ('clasificador', modelo_fase4),
])
```

Pasar ese pipeline sin ajustar a la validación cruzada usando SOLO X_train e y_train.
Cada pliegue aprenderá su propio escalado y categorías. No reutilizar las matrices ni las
copias diagnósticas ajustadas en Fase 3. También se puede comparar candidatos con validación,
manteniendo cualquier ajuste de preprocesamiento en entrenamiento.

Prueba se carga únicamente cuando se haya cerrado la selección del modelo:

```python
X_test, y_test, traza_test = cargar_particion('test', artefactos)
```

El manifiesto y las particiones ya están fijados: no volver a dividir los datos en los notebooks
de cada modelo. Cualquier reentrenamiento final con entrenamiento + validación se decidirá en
Fase 4 antes de evaluar prueba y mantendrá prueba completamente fuera de todo ajuste.

## Límites y decisiones pendientes

- Evaluar macro F1, matriz de confusión y precision/recall/F1 por clase, especialmente High.
  El KPI High global es 24,974 % y no es una métrica predictiva.
- La validación previa usa todos los registros disponibles para una auditoría de calidad y
  continuidad, sin seleccionar modelos con prueba. La Fase 2 exploró el mismo CSV completo:
  prueba no puede considerarse una cohorte externa nunca inspeccionada. La selección de modelos
  debe respetar el bloqueo actual y una validación externa requeriría datos nuevos.
- La publicación original está verificada. Confirmar recolección, escala GPA, instrumentos, momentos de observación y fórmula de etiqueta.
- Evaluar sensibilidad sin GPA final y retención con las mismas particiones. Eso no convierte
  automáticamente al resto de variables en información disponible para detección anticipada.
- Investigar posible circularidad de horas, dependencia, ansiedad u otros predictores con la etiqueta.
- GPA_Change queda como propuesta secundaria, fuera del conjunto principal.
- No aplicar SMOTE, recortes, imputación o selección de variables por defecto. Si se evalúan,
  ajustar cada operación solo dentro de entrenamiento o del pliegue correspondiente.
- One-hot maneja categorías desconocidas con un bloque de ceros. Revisar su frecuencia y
  significado cuando se apliquen estos pipelines a nuevos datos.
"""
(carpeta_artefactos / 'INSTRUCCIONES_FASE4.md').write_text(instrucciones_fase4.strip() + '\n', encoding='utf-8', newline='\n')
archivos_salida = ['train.csv', 'validation.csv', 'test.csv', 'variables.csv', 'decisiones_limpieza.csv',
                  'preprocesamiento.py', 'INSTRUCCIONES_FASE4.md', 'requirements_preparacion.txt']
hashes_salida = {nombre: {'sha256': sha256((carpeta_artefactos / nombre).read_bytes()),
                         'bytes': (carpeta_artefactos / nombre).stat().st_size} for nombre in archivos_salida}
manifiesto = {
    'schema_version': 1, 'crisp_dm_phase': 3,
    'population': 'Estudiantes universitarios representados en el CSV',
    'scope': 'Clasificación multiclase retrospectiva sobre registros completos',
    'input': {'path': RUTA_CSV.as_posix(), 'file': RUTA_CSV.name, 'sha256': hash_entrada,
              'rows': len(df_original), 'columns': len(columnas_originales), 'separator': ';', 'encoding': 'utf-8-sig',
              'class_counts': {c: int(conteos_iniciales[c]) for c in CLASES}, 'high_kpi_percent_exact': str(kpi_inicial)},
    'references': {p.as_posix(): sha256((RAIZ / p).read_bytes()) for p in [Path('README.md'), RUTA_DOC, Path('docs/PROCEDENCIA_DATASET.md'), Path('docs/DICCIONARIO_DATOS.md'), Path('docs/ENTORNO_PROYECTO.md'), Path('requirements.txt'), Path('requirements-lock.txt'), *PREVIOS]},
    'previous_csv_checks': referencias,
    'split': {'seed': SEMILLA, 'proportions': PROPORCIONES, 'stratification': OBJETIVO,
              'procedure': 'train_test_split test_size=0.30; remainder split test_size=0.50; seed=42 in both',
              'ordering': 'Orden devuelto por train_test_split, sin ordenar ni volver a barajar',
              'group_check': 'Student_ID único: no hay grupos repetidos; bloquear división por filas si aparecen',
              'row_id': 'Posición de fila original desde cero; sin cabecera', 'membership_sha256': HUELLAS_PARTICIONES},
    'schema': {'original_columns': columnas_originales,
               'original_dtypes': {c: str(t) for c, t in df_original.dtypes.items()},
               'partition_columns': list(df_elegible.columns), 'predictors': PREDICTORES,
               'numeric': NUMERICAS, 'categorical': CATEGORICAS, 'boolean': BOOLEANAS,
               'target': OBJETIVO, 'target_classes': CLASES, 'target_encoding': 'Etiquetas originales de texto, sin mapeo numérico',
               'trace_columns': TRAZABILIDAD, 'not_included': ['Student_ID', OBJETIVO, '_row_id', 'GPA_Change']},
    'quality': calidad,
    'decisions': decisiones.to_dict(orient='records'),
    'changes': {'records_modified': 0, 'records_excluded': 0, 'eligible_rows': len(df_elegible),
                'added_trace_column': '_row_id', 'high_kpi_percent_after': str(kpi_inicial), 'kpi_change_percentage_points': '0'},
    'preprocessing': {'logistica': 'StandardScaler numérico, OneHotEncoder categórico, booleanos 0/1',
                      'arbol': 'Numéricas sin escalado, OneHotEncoder categórico, booleanos 0/1',
                      'unknown_categories': 'handle_unknown=ignore', 'imputation': None, 'balancing': None,
                      'learned_parameters_exported': False, 'transformed_matrices_exported': False},
    'partitions': {
        nombre: {'file': f'{nombre}.csv', 'rows': len(datos), 'columns': datos.shape[1],
                 'class_counts': {c: int(datos[OBJETIVO].eq(c).sum()) for c in CLASES},
                 'high_kpi_percent': str(Decimal(int(datos[OBJETIVO].eq('High').sum())) * 100 / Decimal(len(datos)))}
        for nombre, datos in particiones.items()},
    'versions': {'Python': sys.version.split()[0], **versiones},
    'limitations': ['Publicación verificada; recolección y carácter sintético no confirmados; sin diagnóstico clínico',
                    'Fórmula de etiqueta desconocida y posible circularidad',
                    'Disponibilidad temporal no acreditada; sin detección anticipada',
                    'Fase 2 exploró el CSV completo: prueba no es una cohorte externa nunca inspeccionada'],
    'provenance': {'url': 'https://www.kaggle.com/datasets/ranaghulamnabi/ai-usage-and-student-academic-performance-analysis', 'publisher': 'ranaghulamnabi', 'version': 1, 'license': 'CC0: Public Domain', 'consultation_date': '2026-10-09', 'original_sha256': '4d911088c4b12d60a450a9acae6b606f4119ebbb48679518e427a4fc00778472'},
    'content_sha256': hash_semantico,
    'artifacts': hashes_salida,
}
(carpeta_artefactos / 'manifest.json').write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2) + '\n',
                                               encoding='utf-8', newline='\n')
mostrar(pd.DataFrame([{'Archivo': (RUTA_ARTEFACTOS / nombre).as_posix(), 'SHA-256': datos['sha256'],
                      'Bytes': datos['bytes']} for nombre, datos in hashes_salida.items()]))
print('Manifiesto:', (RUTA_ARTEFACTOS / 'manifest.json').as_posix())
texto('**Exportación:** las particiones contienen datos sin transformaciones aprendidas. '
      'No se guarda un preprocesador ajustado: Fase 4 creará una instancia nueva por entrenamiento. '
      'Los documentos describen selección, limpieza, tipos y límites; el manifiesto registra las versiones usadas.')


,Archivo,SHA-256,Bytes
0,wrangler/03_PREPARACION_DATOS/artefactos/train.csv,ad1350d1a98ea6cbb8f52d572c930257f5f3010033a0ca9969109d32e15e15fc,4.327.568
1,wrangler/03_PREPARACION_DATOS/artefactos/validation.csv,218af598fa78428fddafd4b530d3563d985a7ada0a2b4ffe758ea2cff3369e0e,926.684
2,wrangler/03_PREPARACION_DATOS/artefactos/test.csv,380f4be4f1d48fed8b44a688bdfc9990654238b9ecb9943f18c2651ecc1fc4b8,927.752
3,wrangler/03_PREPARACION_DATOS/artefactos/variables.csv,33e43397c774e5b474137b338e6e65634573c71b4a35a93164c069d14de12e89,3.116
4,wrangler/03_PREPARACION_DATOS/artefactos/decisiones_limpieza.csv,315eb7368fb3606479952015673b71d301d62a2fc647dd8d3f94fdca324ca062,1.267
5,wrangler/03_PREPARACION_DATOS/artefactos/preprocesamiento.py,c4f370181164f711cb6fa10c36c67e6d7ca082b7be852d9d6958c1efdbf873d5,5.526
6,wrangler/03_PREPARACION_DATOS/artefactos/INSTRUCCIONES_FASE4.md,f2c90303cda08bfb29f962b30620da68bc406a032a84b28989745ea9074aba0e,4.590
7,wrangler/03_PREPARACION_DATOS/artefactos/requirements_preparacion.txt,db94247c199f1e71c7503451c573551df039fbf65cb817603ffc2d7757da8a7f,89


Manifiesto: wrangler/03_PREPARACION_DATOS/artefactos/manifest.json


**Exportación:** las particiones contienen datos sin transformaciones aprendidas. No se guarda un preprocesador ajustado: Fase 4 creará una instancia nueva por entrenamiento. Los documentos describen selección, limpieza, tipos y límites; el manifiesto registra las versiones usadas.

## 7. Verificación de carga y transformaciones sin entrenar clasificadores

Se carga el módulo realmente exportado y se reconstruye cada partición para comprobar igualdad
con el registro de origen. Prueba se usa aquí solo para integridad y distribución, sin ajustar
transformaciones, seleccionar variables ni calcular desempeño.

Las copias diagnósticas se ajustan exclusivamente con entrenamiento y se aplican a validación
y a un registro artificial con categoría desconocida. Son verificaciones de implementación;
no se guardan como entradas de modelado ni se reutilizan en validación cruzada.


In [11]:
especificacion = importlib.util.spec_from_file_location('eduanalytics_preparacion_exportada', carpeta_artefactos / 'preprocesamiento.py')
modulo_exportado = importlib.util.module_from_spec(especificacion)
especificacion.loader.exec_module(modulo_exportado)
cargadas = {nombre: modulo_exportado.cargar_particion(nombre, carpeta_artefactos) for nombre in particiones}
for nombre, (X_cargada, y_cargada, traza_cargada) in cargadas.items():
    reconstruida = pd.concat([X_cargada, y_cargada, traza_cargada], axis=1)[list(df_elegible.columns)].reset_index(drop=True)
    esperada = particiones[nombre].reset_index(drop=True)
    pd.testing.assert_frame_equal(reconstruida, esperada, check_exact=True)
    assert X_cargada.index.equals(y_cargada.index) and X_cargada.index.equals(traza_cargada.index)
    assert list(X_cargada.columns) == PREDICTORES and set(y_cargada) == set(CLASES)
    print(nombre, '| X:', X_cargada.shape, '| y:', y_cargada.shape, '| tipos y valores originales: idénticos')
for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert not set(cargadas[a][2]._row_id) & set(cargadas[b][2]._row_id)
    assert not set(cargadas[a][2].Student_ID) & set(cargadas[b][2].Student_ID)
print('Carga verificada desde los archivos exportados, sin intersecciones entre particiones.')


train | X: (35000, 14) | y: (35000,) | tipos y valores originales: idénticos
validation | X: (7500, 14) | y: (7500,) | tipos y valores originales: idénticos
test | X: (7500, 14) | y: (7500,) | tipos y valores originales: idénticos
Carga verificada desde los archivos exportados, sin intersecciones entre particiones.


In [12]:
X_train, y_train, traza_train = cargadas['train']
X_val, y_val, traza_val = cargadas['validation']
pruebas_transformacion = []
nombres_transformados = {}
for modelo in ['logistica', 'arbol']:
    diagnostico = modulo_exportado.crear_preprocesador(modelo)
    original_sin_ajustar = modulo_exportado.crear_preprocesador(modelo)
    assert diagnostico is not original_sin_ajustar
    try:
        check_is_fitted(diagnostico.named_steps['columnas'])
    except NotFittedError:
        pass
    else:
        raise AssertionError('La fábrica devolvió un preprocesador ajustado.')
    z_train = diagnostico.fit_transform(X_train)
    z_val = diagnostico.transform(X_val)
    assert not hasattr(original_sin_ajustar.named_steps['columnas'], 'transformers_')
    columnas = diagnostico.named_steps['columnas']
    encoder = columnas.named_transformers_['categoricas']
    for c, categorias in zip(CATEGORICAS, encoder.categories_):
        assert set(categorias) == set(X_train[c].unique())
    if modelo == 'logistica':
        scaler = columnas.named_transformers_['numericas'].named_steps['escalado']
        assert int(scaler.n_samples_seen_) == len(X_train)
        np.testing.assert_allclose(scaler.mean_, X_train[NUMERICAS].mean().to_numpy(), atol=1e-12)
        np.testing.assert_allclose(scaler.var_, X_train[NUMERICAS].var(ddof=0).to_numpy(), atol=1e-12)
    else:
        np.testing.assert_array_equal(z_train[:, :len(NUMERICAS)], X_train[NUMERICAS].to_numpy())
    np.testing.assert_array_equal(z_train[:, -1], X_train.Paid_Subscription.to_numpy(dtype=float))
    desconocida = X_train.iloc[[0]].copy(deep=True)
    desconocida['Major_Category'] = '__categoria_nueva_de_verificacion__'
    z_desconocida = diagnostico.transform(desconocida)
    nombres = diagnostico.get_feature_names_out()
    nombres_transformados[modelo] = nombres.tolist()
    posiciones_area = [i for i, nombre in enumerate(nombres) if nombre.startswith('categoricas__Major_Category_')]
    assert z_desconocida[0, posiciones_area].sum() == 0
    assert z_train.shape[1] == z_val.shape[1] == z_desconocida.shape[1] == len(nombres)
    assert np.isfinite(z_train).all() and np.isfinite(z_val).all() and np.isfinite(z_desconocida).all()
    assert not any(OBJETIVO in nombre or 'Student_ID' in nombre or '_row_id' in nombre for nombre in nombres)
    assert not hasattr(clone(diagnostico).named_steps['columnas'], 'transformers_')
    pruebas_transformacion.append({'Modelo previsto': modelo, 'Filas de ajuste': len(X_train),
                                  'Columnas de entrada': X_train.shape[1], 'Columnas transformadas': len(nombres),
                                  'Filas de validación transformadas': len(X_val),
                                  'Salida finita': True, 'Desconocida aceptada': True})
    # Descartar la copia ajustada y sus matrices: la Fase 4 carga solamente registros originales.
    del diagnostico, z_train, z_val, z_desconocida
mostrar(pd.DataFrame(pruebas_transformacion), 0)
mostrar(pd.DataFrame({'Característica transformada': nombres_transformados['logistica']}))
assert nombres_transformados['logistica'] == nombres_transformados['arbol']
texto('**Verificación:** ambos diseños generan 30 columnas con el entrenamiento actual; categorías y '
      'parámetros del escalador coinciden con entrenamiento. La categoría artificial desconocida se representa '
      'con ceros en su bloque, sin modificar un registro real. Se verifican valores, nombres y alineación, '
      'no desempeño predictivo. Las instancias nuevas y los clones permanecen sin ajustar.')


,Modelo previsto,Filas de ajuste,Columnas de entrada,Columnas transformadas,Filas de validación transformadas,Salida finita,Desconocida aceptada
0,logistica,35.000,14,30,7.500,True,True
1,arbol,35.000,14,30,7.500,True,True


,Característica transformada
0,numericas__Pre_Semester_GPA
1,numericas__Weekly_GenAI_Hours
2,numericas__Tool_Diversity
3,numericas__Traditional_Study_Hours
4,numericas__Perceived_AI_Dependency
5,numericas__Anxiety_Level_During_Exams
6,numericas__Post_Semester_GPA
7,numericas__Skill_Retention_Score
8,categoricas__Major_Category_Arts
9,categoricas__Major_Category_Business


**Verificación:** ambos diseños generan 30 columnas con el entrenamiento actual; categorías y parámetros del escalador coinciden con entrenamiento. La categoría artificial desconocida se representa con ceros en su bloque, sin modificar un registro real. Se verifican valores, nombres y alineación, no desempeño predictivo. Las instancias nuevas y los clones permanecen sin ajustar.

## 8. Antes y después, discrepancias y efecto sobre el KPI

La tabla distingue los registros originales y elegibles de la matriz X y de los archivos con
trazabilidad. Reducir columnas de X no significa eliminar estudiantes. Una variación de
proporciones por partición no equivale a reducción real del riesgo.


In [13]:
antes_despues = []
for etapa, datos, etiquetas in [('Original', df_original, df_original[OBJETIVO]),
                               ('Elegible (sin auxiliar)', df_elegible[columnas_originales], y),
                               ('X principal', X, y),
                               *[(nombre + ' (con trazabilidad)', datos, datos[OBJETIVO]) for nombre, datos in particiones.items()]]:
    fila = {'Etapa': etapa, 'Filas': len(datos), 'Columnas': datos.shape[1],
            'Nulos': int(datos.isna().sum().sum()), 'Duplicados completos': int(datos.duplicated().sum()),
            'Duplicados Student_ID': int(datos.Student_ID.duplicated().sum()) if 'Student_ID' in datos else 'No está en X'}
    for c in CLASES:
        fila[c] = int(etiquetas.eq(c).sum())
    fila['High (%)'] = 100 * int(etiquetas.eq('High').sum()) / len(etiquetas)
    antes_despues.append(fila)
mostrar(pd.DataFrame(antes_despues).set_index('Etapa'))
assert conteos_iniciales.equals(df_elegible[OBJETIVO].value_counts().reindex(CLASES))
kpi_final = Decimal(int(df_elegible[OBJETIVO].eq('High').sum())) * 100 / Decimal(len(df_elegible))
assert kpi_final == kpi_inicial
texto(f'**Efecto de preparación:** cero registros corregidos y cero excluidos. KPI global '
      f'**{es(kpi_inicial)} % → {es(kpi_final)} %**, diferencia **{es(kpi_final-kpi_inicial)} puntos porcentuales**. '
      'El conjunto elegible sigue teniendo 16 variables originales; X tiene 14 y los CSV 17 por el auxiliar de trazabilidad. '
      'No se muestra una mejora institucional ni un resultado de clasificación.')

# Recalcular las medias publicadas para conservar la evidencia de discrepancias, sin editar docs.
mapa_doc = {'Horas de IA por semana': 'Weekly_GenAI_Hours', 'Dependencia percibida de IA': 'Perceived_AI_Dependency',
            'Ansiedad durante exámenes': 'Anxiety_Level_During_Exams', 'Horas de estudio tradicional': 'Traditional_Study_Hours',
            'GPA final': 'Post_Semester_GPA', 'Retención de habilidades': 'Skill_Retention_Score'}
bloque_medias = re.search(r'^## 18\..*?(?=^## \d+\.|\Z)', documento, flags=re.M | re.S).group(0)
medias_reales = df_original.groupby(OBJETIVO)[list(mapa_doc.values())].mean()
discrepancias = []
for linea in bloque_medias.splitlines():
    if linea.strip().startswith('|'):
        valores = [v.strip() for v in linea.strip().strip('|').split('|')]
        if valores[0] in mapa_doc:
            c = mapa_doc[valores[0]]
            for clase, publicado in zip(CLASES, valores[1:4]):
                referencia = Decimal(publicado.replace(',', '.'))
                real = Decimal(str(medias_reales.loc[clase, c]))
                redondeado = real.quantize(Decimal('.01'), rounding=ROUND_HALF_UP)
                if redondeado != referencia:
                    discrepancias.append({'Variable': c, 'Grupo': clase, 'Documentado': float(referencia),
                                          'Recalculado': float(real), 'Recalculado a 2 decimales': float(redondeado),
                                          'Diferencia': float(real-referencia)})
mostrar(pd.DataFrame(discrepancias), 6)
texto(f'**Discrepancias actuales:** {len(discrepancias)} medias por grupo. Las tres diferencias históricas se corrigieron en documentación y quedaron registradas en docs §29. Categorías y nombres están alineados; no se modificaron valores del CSV.')
assert not discrepancias


,Filas,Columnas,Nulos,Duplicados completos,Duplicados Student_ID,Low,Medium,High,High (%)
Etapa,,,,,,,,,
Original,50.000,16,0,0,0,16.369,21.144,12.487,"24,974"
Elegible (sin auxiliar),50.000,16,0,0,0,16.369,21.144,12.487,"24,974"
X principal,50.000,14,0,0,No está en X,16.369,21.144,12.487,"24,974"
train (con trazabilidad),35.000,17,0,0,0,11.458,14.801,8.741,"24,974"
validation (con trazabilidad),7.500,17,0,0,0,2.455,3.172,1.873,"24,973"
test (con trazabilidad),7.500,17,0,0,0,2.456,3.171,1.873,"24,973"


**Efecto de preparación:** cero registros corregidos y cero excluidos. KPI global **24,974 % → 24,974 %**, diferencia **0,000 puntos porcentuales**. El conjunto elegible sigue teniendo 16 variables originales; X tiene 14 y los CSV 17 por el auxiliar de trazabilidad. No se muestra una mejora institucional ni un resultado de clasificación.

**Discrepancias actuales:** 0 medias por grupo. Las tres diferencias históricas se corrigieron en documentación y quedaron registradas en docs §29. Categorías y nombres están alineados; no se modificaron valores del CSV.

## 9. Traspaso y decisiones que debe revisar la Fase 4

1. Cargar las particiones guardadas, conservar su pertenencia y construir un pipeline completo
   de preprocesamiento más clasificador. No volver a dividir el CSV para cada modelo.
2. Ajustar categorías, escalado y cualquier transformación aprendida dentro de entrenamiento
   o del pliegue correspondiente. No entregar matrices ya transformadas a validación cruzada.
3. Seleccionar modelos con entrenamiento/validación. Reservar prueba para evaluación final.
   La Fase 2 exploró el CSV completo: el conjunto de prueba fijado ahora no es una cohorte externa
   nunca inspeccionada. Este límite no autoriza nuevas decisiones de modelado con prueba.
4. Evaluar métricas por clase, macro F1 y especial interés en High. El KPI descriptivo permanece separado.
5. Investigar procedencia, licencia, simulación, fórmula de la etiqueta, instrumentos y escalas.
   Una buena clasificación interna podría reproducir reglas del generador.
6. Evaluar sensibilidad sin GPA final y retención con las mismas particiones. Revisar también
   la relación de horas, dependencia y ansiedad con la generación de la etiqueta. Una exclusión
   no demuestra por sí sola disponibilidad anticipada del resto de campos.
7. Conservar extremos plausibles; cualquier recorte, imputación, codificación alternativa,
   selección o balanceo que se evalúe deberá aprenderse únicamente dentro del entrenamiento.
8. Mantener `GPA_Change` como propuesta académica secundaria, fuera del conjunto principal actual.

Instrucciones completas y código de carga: [INSTRUCCIONES_FASE4.md](artefactos/INSTRUCCIONES_FASE4.md).
No se entrenaron clasificadores ni se calcularon predicciones, matrices de confusión o métricas de desempeño.


In [14]:
# Integridad final: fuente y referencias intactas; particiones y documentos coherentes.
assert sha256((RAIZ / RUTA_CSV).read_bytes()) == hash_entrada
assert np.array_equal(pd.util.hash_pandas_object(df_original, index=True).values, firma_original)
for referencia, huella in manifiesto['references'].items():
    assert sha256((RAIZ / referencia).read_bytes()) == huella
for archivo, info in manifiesto['artifacts'].items():
    assert sha256((carpeta_artefactos / archivo).read_bytes()) == info['sha256']
assert not manifiesto['preprocessing']['learned_parameters_exported']
assert not manifiesto['preprocessing']['transformed_matrices_exported']
texto(f'**Fase 3 ejecutada:** `{RUTA_NOTEBOOK.as_posix()}`. Entradas en `{RUTA_ARTEFACTOS.as_posix()}`; '
      '35.000 / 7.500 / 7.500 registros, 14 predictores y objetivo textual multiclase. '
      f'KPI global **{es(kpi_final)} %**, sin cambios. Se verificaron integridad, carga, particiones '
      'y transformaciones con ajuste exclusivo en entrenamiento. El modelado queda para Fase 4.')
print('CSV, documentos y notebooks previos: hashes intactos.')
print('Intersecciones vacías, cobertura completa y carga exacta: comprobadas.')
print('Artefactos oficiales:', len(manifiesto['artifacts']) + 1, '(incluido manifest.json)')


**Fase 3 ejecutada:** `wrangler/03_PREPARACION_DATOS/03_preparacion_datos.ipynb`. Entradas en `wrangler/03_PREPARACION_DATOS/artefactos`; 35.000 / 7.500 / 7.500 registros, 14 predictores y objetivo textual multiclase. KPI global **24,974 %**, sin cambios. Se verificaron integridad, carga, particiones y transformaciones con ajuste exclusivo en entrenamiento. El modelado queda para Fase 4.

CSV, documentos y notebooks previos: hashes intactos.
Intersecciones vacías, cobertura completa y carga exacta: comprobadas.
Artefactos oficiales: 9 (incluido manifest.json)
